# 02 — Check the N / P₂O₅ supply numbers

Reads the output of a full run:

```bash
python3 scripts/map_buildings.py                       # -> data/processed/task1/run_<date>/
python3 scripts/compute_supply.py data/processed/task1/run_<date>
```

and checks every number by hand. Formulas (`geo_anom/task1/supply.py`):

* **farm N** = registry headcount × N per head per flock × flocks per year
* **building N** = farm N × (building floor area ÷ farm's detected floor area)

Headcount is the registry's *permitted capacity*, not what the imagery shows. Imagery decides **where** the supply sits and how it is split across barns.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
%matplotlib inline
%load_ext autoreload
%autoreload 2   # pick up edits to geo_anom/ without restarting the kernel
pd.set_option("display.width", 200, "display.max_columns", 30, "display.max_rows", 100)

from geo_anom.core.config import get_config
from geo_anom.task1.supply import AWTF_BENCHMARKS, COEFF_KEY, farm_supply, building_supply, species_totals

In [ ]:
from geo_anom.task1.pipeline import latest_run
RUN = latest_run()     # or ROOT / "data/processed/task1/run_<date>"
print(json.dumps(json.loads((RUN / "run_meta.json").read_text()), indent=2))

## 1. The coefficients, and a sanity check against an independent number

The AWTF report (Lansing et al., 2023, p. 23) estimates **21.85 million lb N from broiler litter in Maryland in 2019**, from ~270–300 million birds/yr. That implies roughly **0.07–0.08 lb N per bird marketed**. Compare with what our config uses per bird per flock:

In [ ]:
cfg = get_config()
coeffs = pd.DataFrame({k: vars(v) for k, v in cfg.nutrient_coefficients.items()}).T
coeffs["N_per_year_per_head_of_capacity"] = coeffs["N_lbs_per_head_per_year"] * coeffs["flocks_per_year"]
display(coeffs)

awtf_n_per_bird = AWTF_BENCHMARKS["broiler_N_lbs_2019"] / 285e6
print(f"AWTF-implied broiler N per bird: {awtf_n_per_bird:.3f} lb")
print(f"config broiler N per bird per flock: {cfg.nutrient_coefficients['broiler_chicken'].N_lbs_per_head_per_year} lb "
      f"-> {cfg.nutrient_coefficients['broiler_chicken'].N_lbs_per_head_per_year / awtf_n_per_bird:.0f}x higher")

> **Read this before using any total.** If the ratio above is far from 1, every broiler N/P number downstream is off by that factor. The coefficient (and whether it is per bird per flock or per bird-space per year) is a domain question for Stephanie's lab — the AWTF Table 1a.1 coefficient set is the obvious reference.

## 2. Recompute one farm by hand

In [ ]:
sites = json.loads((RUN / "sites.json").read_text())
buildings = __import__("geopandas").read_file(RUN / "buildings.geojson")
farms = farm_supply(sites, buildings)
bsup = building_supply(buildings, farms)

FARM = "Vy Tran"
f = farms[farms.farm_name.str.contains(FARM, case=False)].iloc[0]
c = cfg.nutrient_coefficients[COEFF_KEY[f.animal_type]]
by_hand = f.headcount * c.N_lbs_per_head_per_year * c.flocks_per_year
print(f"{f.farm_name}: {f.headcount:,} head x {c.N_lbs_per_head_per_year} lb x {c.flocks_per_year} flocks = {by_hand:,.0f} lb N/yr")
print(f"pipeline: {f.annual_N_lbs:,.0f}  -> match: {abs(by_hand - f.annual_N_lbs) < 1}")

mine = bsup[bsup.assigned_site_id == f.site_id]
print(f"{len(mine)} buildings; their N sums to {mine.annual_N_lbs.sum():,.0f} (should equal the farm total)")
display(mine[["building_id", "area_m2", "area_share", "annual_N_lbs", "assigned_distance_m", "n_detections"]])

## 3. Statewide totals by species, against the AWTF species split

AWTF (2019, all sources, not just CAFO permits): broilers **51%** of combined N+P, all cattle **36%**. The CAFO registry is mostly poultry, so a registry-only total is expected to over-weight broilers — how far off are we?

In [ ]:
totals = species_totals(farms)
display(totals)
print("located on detected buildings:", int(farms.located_by.eq("detected_buildings").sum()), "farms;",
      "registry point only:", int(farms.located_by.eq("registry_point_only").sum()))
print("no coefficient (zero supply):", sorted(farms.loc[farms.no_coefficient, "animal_type"].astype(str).unique()))

In [ ]:
if "broiler" in totals.index and len(farms) >= 400:
    ours = totals.loc["broiler", "annual_N_lbs"]
    print(f"broiler N: {ours:,.0f} here vs {AWTF_BENCHMARKS['broiler_N_lbs_2019']:,.0f} AWTF 2019 -> {ours / AWTF_BENCHMARKS['broiler_N_lbs_2019']:.1f}x")

## 4. Where is the supply that the imagery can't place?

Farms whose supply sits only at the registry point (no detected building) — mostly dairy. Sorted by N.

In [ ]:
farms[farms.located_by == "registry_point_only"].sort_values("annual_N_lbs", ascending=False)[
    ["farm_name", "county", "animal_type", "headcount", "annual_N_lbs"]].head(25)